## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pathlib import Path
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
import os


In [2]:
MODEL = "gpt-4.1-nano"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [25]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [26]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

### These LangChain objects implement the method `invoke()`

In [27]:
retriever.invoke("Who is alex?")

[Document(id='e8002bc3-0598-4f17-ac2c-32fc37a0351c', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Alex Harper.md'}, page_content='- **Awards**:  \n  - Insurellm "SDR of the Year" Award (2022)  \n  - Monthly MVP Recognition (3 times in 2023)  \n\n- **Interests**:  \n  - In Alex\'s spare time, they enjoy participating in community volunteer programs, particularly those focused on financial literacy.  \n  - Alex is also an avid runner and has participated in several charity marathons.'),
 Document(id='e0566cc0-2451-4ac1-a76b-a64e9073ecbf', metadata={'source': 'knowledge-base/employees/Alex Thomson.md', 'doc_type': 'employees'}, page_content='--- \n**Comment:** Alex Thomson is considered a cornerstone of Insurellm’s sales team and has a bright future within the organization.'),
 Document(id='ecedfcb2-a8cd-442e-9ef0-d5687e1af034', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Alex Thomson.md'}, page_content='## Other HR Notes\n- Alex Thomso

In [28]:
llm.invoke("Who is Avery Lancaster?")

AIMessage(content='As of my knowledge cutoff in October 2023, there is no widely known or publicly prominent individual named Avery Lancaster. If you have more context or specific details about this person, please provide them, and I will do my best to assist you.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 50, 'prompt_tokens': 12, 'total_tokens': 62, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-nano-2025-04-14', 'system_fingerprint': 'fp_71b24f5eef', 'id': 'chatcmpl-EWn2hXOj6ptvtlch6zq9h7uInNiYU', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--f52c3041-aeae-4660-a45b-ce68dbd31d4f-0', usage_metadata={'input_tokens': 12, 'output_tokens': 50, 'total_tokens': 62, 'input_token_details': {'audio':

## Time to put this together!

In [29]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If the context doesn't contain the answer, say you don't know.

Context:
{context}
"""

In [30]:

def person_of(doc):
    return Path(doc.metadata["source"]).stem  
docs = retriever.invoke("who is alex")
print(docs)


[Document(id='e0566cc0-2451-4ac1-a76b-a64e9073ecbf', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Alex Thomson.md'}, page_content='--- \n**Comment:** Alex Thomson is considered a cornerstone of Insurellm’s sales team and has a bright future within the organization.'), Document(id='e8002bc3-0598-4f17-ac2c-32fc37a0351c', metadata={'source': 'knowledge-base/employees/Alex Harper.md', 'doc_type': 'employees'}, page_content='- **Awards**:  \n  - Insurellm "SDR of the Year" Award (2022)  \n  - Monthly MVP Recognition (3 times in 2023)  \n\n- **Interests**:  \n  - In Alex\'s spare time, they enjoy participating in community volunteer programs, particularly those focused on financial literacy.  \n  - Alex is also an avid runner and has participated in several charity marathons.'), Document(id='ecedfcb2-a8cd-442e-9ef0-d5687e1af034', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Alex Thomson.md'}, page_content='## Other HR Notes\n- Alex Thomson 

In [31]:

def answer_question(question: str, history):
    
    docs = retriever.invoke(question)
    # context = "\n\n".join(doc.page_content for doc in docs)
    context = "\n\n".join(f"[Person: {person_of(d)}]\n{d.page_content}" for d in docs)
    messages = [SystemMessage(content=SYSTEM_PROMPT_TEMPLATE.format(context=context))]
    for user_msg, ai_msg in history:
        messages += [HumanMessage(content=user_msg), AIMessage(content=ai_msg)]
    messages.append(HumanMessage(content=question))
    response = llm.invoke(messages)
    return response.content

In [32]:
answer_question("Who is Alex Thompson", [])

"Alex Thomson is a Sales Development Representative at Insurellm, based in Austin, Texas. He is recognized as a key member of the company's sales team, with a bright future ahead. Alex is also actively involved in the company's Diversity and Inclusion committee and has participated in community outreach programs. He has received external training on advanced CRM usage and is committed to professional development, including plans to pursue certification in Sales Enablement in 2024."

## What could possibly come next? 😂

In [33]:
gr.ChatInterface(answer_question).launch()

/Users/lalith/Developer/Code/llmprojects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!